In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from google import genai

client = genai.Client()

In [3]:
def llm(prompt):
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )
    return response.text


In [4]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [5]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [6]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [7]:
question = "I just discovered the course. Can I join now?"


def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [8]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [9]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [10]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [11]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [12]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [15]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [31]:
response = client.models.generate_content(
    model="gemini-2.5-flash", 
    contents=prompt
)


In [32]:
print(response.text)

Yes, you can join now and start the course.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Certificates are only awarded if you finish the course with a "live" cohort, which includes submitting your project and participating in peer-reviews when the course is actively running.

You can start learning whenever you want, as the videos and GitHub materials are available. There's no formal registration process required to start learning; you can just dive in. Make sure to check the deadlines on the course management platform if you intend to submit homework and aim for a certificate.


In [ ]:
from google.genai import types

message_history = [
    types.Content(
        role="user",
        parts=[types.Part.from_text(text=prompt)]
    )
]
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=message_history,
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCTIONS,
    ),
)

In [41]:
print(response.text)

Yes, you can join now. The videos and GitHub materials are available, and you can start whenever you want. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [46]:
def llm(instructions: str, user_prompt: str, model: str = "gemini-2.5-flash") -> str:
    message_history = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_prompt)]
        )
    ]
    
    config = types.GenerateContentConfig(
        system_instruction=instructions,
    )

    response = client.models.generate_content(
        model=model,
        contents=message_history,
        config=config
    )

    return response.text

In [47]:
def rag(query, model="gemini-2.5-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [48]:
answer = rag(question)
print(answer)

Yes, you can join now. You can start whenever you want, as the videos and GitHub materials are available. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
